# Регрессия SI

Прогнозирование селективного индекса (SI = CC50/IC50)


In [1]:
import numpy as np
import pickle
RANDOM_STATE = 42
ASSETS_DIR = 'assets'
try:
    import seaborn as sns
    sns.set_style('whitegrid')
except ImportError:
    pass


In [2]:
import os
os.makedirs(ASSETS_DIR, exist_ok=True)
# Загрузка очищенных данных и результатов моделей
with open('_data.pkl', 'rb') as f:
    data = pickle.load(f)
df = data['df']
feat_cols = data['feat_cols']
target_cols = data['target_cols']
with open('_all_results.pkl', 'rb') as f:
    results = pickle.load(f)
stats = results['stats']
reg_results = results['reg_results']
print(f'Данные: {df.shape}, Признаки: {len(feat_cols)}')


Данные: (1001, 195), Признаки: 192


In [3]:
target = 'SI'
y = df[target]
use_log = stats[target]['skew'] > 1.5
y_use = np.log1p(y) if use_log else y.copy()
print(f'Целевая: {target}')
print(f'Лог-преобразование: {use_log} (skew={stats[target]["skew"]:.2f})')
print(f'Исходные: mean={y.mean():.2f}, median={y.median():.2f}, skew={y.skew():.2f}')


Целевая: SI
Лог-преобразование: True (skew=18.01)
Исходные: mean=72.51, median=3.85, skew=18.01


In [4]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
X_all = df[feat_cols].values
scaler = StandardScaler()
Xs = scaler.fit_transform(X_all)


In [5]:
models = [
    ('DummyMean', DummyRegressor, {'strategy': 'mean'}),
    ('DummyMedian', DummyRegressor, {'strategy': 'median'}),
    ('Ridge', Ridge, {'alpha': 1.0}),
    ('Lasso', Lasso, {'alpha': 0.01, 'max_iter': 5000}),
    ('KNN', KNeighborsRegressor, {'n_neighbors': 5}),
    ('RF', RandomForestRegressor, {'n_estimators': 100, 'random_state': RANDOM_STATE}),
    ('HGB', HistGradientBoostingRegressor, {'max_iter': 200, 'max_depth': 6, 'random_state': RANDOM_STATE}),
]

print(f'Модели для {target}:')
for name, ModelClass, kwargs in models:
    m = ModelClass(**kwargs)
    m.fit(Xs, y_use)
    preds = m.predict(Xs)
    mae_cv = cross_val_score(m, Xs, y_use, cv=kf, scoring='neg_mean_absolute_error')
    r2_cv = cross_val_score(m, Xs, y_use, cv=kf, scoring='r2')
    rmse_cv = np.sqrt(-cross_val_score(m, Xs, y_use, cv=kf, scoring='neg_mean_squared_error'))
    print(f'  {name}: MAE={-np.mean(mae_cv):.3f}(+/-{np.std(mae_cv):.3f}), R2={np.mean(r2_cv):.4f}, RMSE={np.mean(rmse_cv):.3f}')


Модели для SI:
  DummyMean: MAE=1.143(+/-0.069), R2=-0.0031, RMSE=1.452
  DummyMedian: MAE=1.096(+/-0.071), R2=-0.1082, RMSE=1.526
  Ridge: MAE=1.024(+/-0.051), R2=0.1225, RMSE=1.348
  Lasso: MAE=1.008(+/-0.060), R2=0.1705, RMSE=1.316
  KNN: MAE=0.962(+/-0.070), R2=0.1599, RMSE=1.329
  RF: MAE=0.887(+/-0.051), R2=0.3275, RMSE=1.186
  HGB: MAE=0.911(+/-0.045), R2=0.2726, RMSE=1.233


In [6]:
best = max(reg_results[target], key=lambda k: reg_results[target][k]['R2_cv'])
r = reg_results[target][best]
print(f'\nЛучшая модель CV для SI: {best}')
print(f'  R2 CV: {r["R2_cv"]:.4f} (+/-{r["R2_cv_std"]:.4f})')
print(f'  MAE CV: {r["MAE_cv"]:.3f} (+/-{r["MAE_cv_std"]:.3f})')
print(f'  RMSE CV: {r["RMSE_cv"]:.3f} (+/-{r["RMSE_cv_std"]:.3f})')



Лучшая модель CV для SI: RF
  R2 CV: 0.3276 (+/-0.0451)
  MAE CV: -0.885 (+/-0.051)
  RMSE CV: 1.186 (+/-0.069)
